# Move selection model
I'm going to start to make a ResNet/CNN model, probably for training or I will use Kaggle or I'll wait the computer to work again

In [1]:
from MLChess import dataset_creation_graph, GraphAndPoolingChessGCN
import torch.nn as nn
import torch
from torch_geometric.data import Batch
from tqdm import tqdm

## Creation of the dataset

In [2]:
training, val, test = dataset_creation_graph("over_mate_1_tactic_evals.csv")

Dataset diviso in:
  Training: 1689648 campioni (70.0%)
  Validation: 362068 campioni (15.0%)
  Test: 362068 campioni (15.0%)
Dataset subset: 1689648 posizioni
Dataset subset: 362068 posizioni
Dataset subset: 362068 posizioni


In [3]:
class TestModelGCN(nn.Module):
    def __init__(self, hidden_dim: int = 256):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.GCN = GraphAndPoolingChessGCN(hidden_dim=self.hidden_dim)

        self.choose_arch = nn.Sequential(
            nn.Linear(3*self.hidden_dim, self.hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=0.25),
            nn.Linear(self.hidden_dim, 1),
        )

        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim*3 + hidden_dim//2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

    
    def forward(self, data):

        graph, combined = self.GCN(data.x, data.edge_index, data.batch, data.global_features)

        src, dst = data.edge_index
        edge_emb = torch.cat([graph[src], graph[dst], graph[dst]-graph[src]], dim=1)

        logits = self.choose_arch(edge_emb).squeeze(-1)
        if hasattr(data, "legal_edge_mask"):
            logits = logits.masked_fill(
                data.legal_edge_mask == 0, -1e9
            )

        value = self.value_head(combined)

        return logits, value

In [4]:
def policy_loss_fn(policy_logits, policy_target, legal_edge_mask):
    # Applica mask dura
    masked_logits = policy_logits.masked_fill(
        legal_edge_mask == 0, -1e9
    )

    log_probs = torch.log_softmax(masked_logits, dim=0)

    loss = -(policy_target * log_probs).sum()
    return loss


In [5]:
def train_step(model, optimizer, batch, device, lambda_policy=1.0, lambda_value=1.0):
    model.train()
    optimizer.zero_grad()
    

    # PyG batch
    data = Batch.from_data_list(batch).to(device)

    policy_logits, value = model(data)

    # --- POLICY LOSS ---
    policy_loss = policy_loss_fn(
        policy_logits,
        data.y_policy,
        data.legal_edge_mask
    )

    # --- VALUE LOSS ---
    value_loss = nn.MSELoss()
    value_loss = value_loss(
        value.squeeze(-1),
        data.y
    )

    loss = lambda_policy * policy_loss + lambda_value * value_loss

    loss.backward()
    optimizer.step()

    return {
        "loss": loss.item(),
        "policy_loss": policy_loss.item(),
        "value_loss": value_loss.item()
    }


In [ ]:
num_epochs = 10
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = TestModelGCN(hidden_dim=256).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in tqdm(range(num_epochs), total = num_epochs):
    epoch_loss = 0.0
    n_batches = 0

    for batch in training.get_batch_iterator(shuffle=True):
        stats = train_step(model, optimizer, batch, device = device)
        epoch_loss += stats["loss"]
        n_batches += 1

    print(
        f"Epoch {epoch:03d} | "
        f"loss={epoch_loss / n_batches:.4f}"
    )


cuda


 10%|█         | 1/10 [24:48<3:43:17, 1488.63s/it]

Epoch 000 | loss=0.2432


 20%|██        | 2/10 [49:22<3:17:20, 1480.01s/it]

Epoch 001 | loss=0.2250


...

 50%|█████     | 5/10 [2:03:11<2:03:05, 1477.03s/it]

Epoch 004 | loss=0.2008


 60%|██████    | 6/10 [2:27:45<1:38:24, 1476.11s/it]

Epoch 005 | loss=0.1956


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f6e3a777a10>>
Traceback (most recent call last):
  File "/home/Koro/miniconda3/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 
